# Modelo-07 — Processamento para o CatBoostClassifier

## Objetivo

Este notebook transforma a tabela mestra diária do sistema PSD em uma base pronta para modelagem. A unidade permanece **estação × plataforma × dia**, identificada por `id_psd` e `data_referencia`.

Nesta etapa serão realizados:

1. carregamento e validação da tabela mestra;
2. criação do alvo de ocorrência de falha nos próximos 7 dias;
3. criação de features históricas orientadas pelas hipóteses da Sprint 1;
4. aplicação de deslocamentos temporais para evitar vazamento do alvo;
5. definição da separação temporal entre treino, validação e teste;
6. exportação da base processada para o notebook de treinamento.

> A coluna `qtd_falhas_confirmadas` será utilizada para construir o alvo. Ela não poderá ser usada diretamente como feature contemporânea do modelo.

## Momento da previsão e janela do alvo

A previsão será realizada **ao final do dia de referência `D`**. Portanto:

- as features poderão utilizar informações conhecidas até o encerramento de `D`;
- o alvo será igual a `1` quando houver ao menos uma falha confirmada entre `D+1` e `D+7`;
- o alvo será igual a `0` quando não houver falha confirmada nessa janela futura;
- falhas ocorridas no próprio dia `D` não farão parte do alvo daquela linha;
- nenhuma informação registrada após `D` poderá ser utilizada como feature.

Representação da janela:

```text
Histórico disponível ───────── D │ D+1  D+2  D+3  D+4  D+5  D+6  D+7
                                │ └──────── janela do alvo ────────┘
                         previsão ao final de D
```

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semente: {RANDOM_SEED}")

Pandas: 3.0.5
NumPy: 2.5.1
Semente: 42


In [3]:
CAMINHO_TABELA_MESTRA = Path("../../data/processados/modelo-07/tabela_mestra_psd.csv")

tabela_mestra = pd.read_csv(
    CAMINHO_TABELA_MESTRA,
    parse_dates=["data_referencia"],
)

print(f"Linhas: {tabela_mestra.shape[0]:,}")
print(f"Colunas: {tabela_mestra.shape[1]}")
display(tabela_mestra.head())

Linhas: 24,772
Colunas: 26


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas,qtd_corretivas_programadas,qtd_indisponibilidades_programadas,duracao_total_programadas_horas,duracao_maxima_programada_horas,qtd_preventivas,qtd_preventivas_canceladas,qtd_preventivas_nao_canceladas,duracao_total_preventivas_horas,duracao_maxima_preventiva_horas
0,2023-02-27,BUT_P1,BUT,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
1,2023-02-27,BUT_P2,BUT,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
2,2023-02-27,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,1,1,1,0.98,0.98,0,0,0.00,0.00,0,0,0,0.00,0.00
3,2023-02-27,FAL_P2,FAL,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
4,2023-02-27,FRA_P1,FRA,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00


In [4]:
tabela_mestra["id_psd"] = tabela_mestra["id_psd"].astype("string")
tabela_mestra["estacao_padrao"] = tabela_mestra["estacao_padrao"].astype("string")
tabela_mestra["plataforma_padrao"] = tabela_mestra["plataforma_padrao"].astype("Int8")

chaves = ["id_psd", "data_referencia"]
tabela_mestra = tabela_mestra.sort_values(chaves).reset_index(drop=True)
intervalos = tabela_mestra.groupby("id_psd")["data_referencia"].diff().dropna()
dias_por_psd = tabela_mestra.groupby("id_psd")["data_referencia"].nunique()

assert not tabela_mestra.duplicated(chaves).any()
assert not tabela_mestra.isna().any().any()
assert intervalos.eq(pd.Timedelta(days=1)).all()
assert dias_por_psd.nunique() == 1
assert (tabela_mestra["qtd_falhas_confirmadas"] <= tabela_mestra["qtd_corretivas_emergenciais"]).all()

auditoria_entrada = pd.DataFrame({
    "verificacao": ["linhas", "colunas", "sistemas_psd", "dias_por_psd", "duplicadas", "nulos"],
    "resultado": [
        len(tabela_mestra),
        tabela_mestra.shape[1],
        tabela_mestra["id_psd"].nunique(),
        int(dias_por_psd.iloc[0]),
        int(tabela_mestra.duplicated(chaves).sum()),
        int(tabela_mestra.isna().sum().sum()),
    ],
})

display(auditoria_entrada)

,verificacao,resultado
0,linhas,24772
1,colunas,26
2,sistemas_psd,22
3,dias_por_psd,1126
4,duplicadas,0
5,nulos,0


In [7]:
HORIZONTE_ALVO_DIAS = 7
grupo_falhas = tabela_mestra.groupby("id_psd", sort=False)["qtd_falhas_confirmadas"]

falhas_por_dia_futuro = pd.concat(
    [grupo_falhas.shift(-dia) for dia in range(1, HORIZONTE_ALVO_DIAS + 1)],
    axis=1,
)
falhas_por_dia_futuro.columns = [f"D+{dia}" for dia in range(1, HORIZONTE_ALVO_DIAS + 1)]

tabela_mestra["qtd_falhas_proximos_7d"] = falhas_por_dia_futuro.sum(
    axis=1, min_count=HORIZONTE_ALVO_DIAS
)

rotulo_disponivel = tabela_mestra["qtd_falhas_proximos_7d"].notna()
tabela_mestra["target_falha_7d"] = pd.Series(pd.NA, index=tabela_mestra.index, dtype="Int8")
tabela_mestra.loc[rotulo_disponivel, "target_falha_7d"] = (
    tabela_mestra.loc[rotulo_disponivel, "qtd_falhas_proximos_7d"] > 0
).astype("Int8")

linhas_sem_rotulo_esperadas = tabela_mestra["id_psd"].nunique() * HORIZONTE_ALVO_DIAS
assert int(tabela_mestra["target_falha_7d"].isna().sum()) == linhas_sem_rotulo_esperadas

distribuicao_alvo = (
    tabela_mestra["target_falha_7d"]
    .value_counts(dropna=False)
    .rename_axis("target_falha_7d")
    .reset_index(name="linhas")
)
distribuicao_alvo["percentual"] = (distribuicao_alvo["linhas"] / len(tabela_mestra) * 100).round(2)

display(distribuicao_alvo)

,target_falha_7d,linhas,percentual
0,0,12834,51.81
1,1,11784,47.57
2,<NA>,154,0.62


In [8]:
primeira_falha = tabela_mestra.loc[
    tabela_mestra["qtd_falhas_confirmadas"] > 0,
    ["id_psd", "data_referencia"],
].iloc[0]

id_psd_exemplo = primeira_falha["id_psd"]
data_falha_exemplo = primeira_falha["data_referencia"]

auditoria_exemplo_alvo = tabela_mestra.loc[
    (tabela_mestra["id_psd"] == id_psd_exemplo)
    & tabela_mestra["data_referencia"].between(
        data_falha_exemplo - pd.Timedelta(days=8),
        data_falha_exemplo + pd.Timedelta(days=2),
    ),
    [
        "data_referencia",
        "id_psd",
        "qtd_falhas_confirmadas",
        "qtd_falhas_proximos_7d",
        "target_falha_7d",
    ],
]

print(f"Falha usada no exemplo: {id_psd_exemplo} em {data_falha_exemplo.date()}")
display(auditoria_exemplo_alvo)

Falha usada no exemplo: BUT_P1 em 2023-03-13


,data_referencia,id_psd,qtd_falhas_confirmadas,qtd_falhas_proximos_7d,target_falha_7d
6,2023-03-05,BUT_P1,0,0.00,0
7,2023-03-06,BUT_P1,0,1.00,1
8,2023-03-07,BUT_P1,0,1.00,1
9,2023-03-08,BUT_P1,0,1.00,1
10,2023-03-09,BUT_P1,0,2.00,1
11,2023-03-10,BUT_P1,0,2.00,1
12,2023-03-11,BUT_P1,0,2.00,1
13,2023-03-12,BUT_P1,0,2.00,1
14,2023-03-13,BUT_P1,1,1.00,1
15,2023-03-14,BUT_P1,0,4.00,1


In [9]:
NOMES_DIAS_SEMANA = {
    0: "segunda",
    1: "terca",
    2: "quarta",
    3: "quinta",
    4: "sexta",
    5: "sabado",
    6: "domingo",
}

numero_dia_semana = tabela_mestra["data_referencia"].dt.dayofweek
tabela_mestra["dia_semana"] = numero_dia_semana.map(NOMES_DIAS_SEMANA).astype("string")
tabela_mestra["domingo_flag"] = numero_dia_semana.eq(6).astype("Int8")

auditoria_dia_semana = (
    tabela_mestra.loc[tabela_mestra["target_falha_7d"].notna()]
    .groupby("dia_semana", as_index=False)
    .agg(
        linhas=("target_falha_7d", "size"),
        taxa_target_positivo=("target_falha_7d", "mean"),
    )
)
auditoria_dia_semana["taxa_target_positivo"] *= 100

display(auditoria_dia_semana.sort_values("taxa_target_positivo", ascending=False))

,dia_semana,linhas,taxa_target_positivo
6,terca,3520,48.64
4,segunda,3520,48.15
1,quarta,3520,48.07
5,sexta,3520,47.73
2,quinta,3520,47.61
0,domingo,3498,47.51
3,sabado,3520,47.36


In [10]:
JANELA_ALARMES_DIAS = 7
tabela_mestra["qtd_criticas_pico_dia"] = (
    tabela_mestra["qtd_criticas_pico_manha"]
    + tabela_mestra["qtd_criticas_pico_tarde"]
)
tabela_mestra["dia_com_anomalia_critica"] = (
    tabela_mestra["qtd_anomalias_criticas_30s"] > 0
).astype("Int8")

grupo_psd = tabela_mestra.groupby("id_psd", sort=False)

tabela_mestra["qtd_anomalias_criticas_30s_7d"] = grupo_psd["qtd_anomalias_criticas_30s"].transform(
    lambda serie: serie.rolling(JANELA_ALARMES_DIAS, min_periods=JANELA_ALARMES_DIAS).sum()
)
tabela_mestra["qtd_criticas_pico_7d"] = grupo_psd["qtd_criticas_pico_dia"].transform(
    lambda serie: serie.rolling(JANELA_ALARMES_DIAS, min_periods=JANELA_ALARMES_DIAS).sum()
)
tabela_mestra["dias_com_anomalia_critica_7d"] = grupo_psd["dia_com_anomalia_critica"].transform(
    lambda serie: serie.rolling(JANELA_ALARMES_DIAS, min_periods=JANELA_ALARMES_DIAS).sum()
)
tabela_mestra["duracao_maxima_alarme_7d"] = grupo_psd["duracao_maxima_alarme_seg"].transform(
    lambda serie: serie.rolling(JANELA_ALARMES_DIAS, min_periods=JANELA_ALARMES_DIAS).max()
)

denominador_criticas = tabela_mestra["qtd_anomalias_criticas_30s_7d"]
tabela_mestra["proporcao_criticas_pico_7d"] = (
    tabela_mestra["qtd_criticas_pico_7d"] / denominador_criticas.replace(0, np.nan)
)
tabela_mestra.loc[denominador_criticas.eq(0), "proporcao_criticas_pico_7d"] = 0.0

colunas_h1_h5 = [
    "qtd_anomalias_criticas_30s_7d",
    "qtd_criticas_pico_7d",
    "dias_com_anomalia_critica_7d",
    "duracao_maxima_alarme_7d",
    "proporcao_criticas_pico_7d",
]

print(f"Linhas sem histórico completo de 7 dias: {tabela_mestra[colunas_h1_h5].isna().any(axis=1).sum():,}")
display(tabela_mestra[["data_referencia", "id_psd", *colunas_h1_h5]].dropna().head())

Linhas sem histórico completo de 7 dias: 132


,data_referencia,id_psd,qtd_anomalias_criticas_30s_7d,qtd_criticas_pico_7d,dias_com_anomalia_critica_7d,duracao_maxima_alarme_7d,proporcao_criticas_pico_7d
6,2023-03-05,BUT_P1,0.00,0.00,0.00,0.00,0.00
7,2023-03-06,BUT_P1,0.00,0.00,0.00,0.00,0.00
8,2023-03-07,BUT_P1,0.00,0.00,0.00,0.00,0.00
9,2023-03-08,BUT_P1,0.00,0.00,0.00,0.00,0.00
10,2023-03-09,BUT_P1,0.00,0.00,0.00,0.00,0.00


In [11]:
def calcular_dias_desde_evento(df, coluna_evento):
    datas_com_evento = df["data_referencia"].where(df[coluna_evento] > 0)
    ultima_data_evento = datas_com_evento.groupby(df["id_psd"]).ffill()
    return (df["data_referencia"] - ultima_data_evento).dt.days.astype("Float64")

tabela_mestra["dias_desde_ultima_corretiva"] = calcular_dias_desde_evento(
    tabela_mestra, "qtd_corretivas_emergenciais"
)
tabela_mestra["dias_desde_ultima_preventiva"] = calcular_dias_desde_evento(
    tabela_mestra, "qtd_preventivas_nao_canceladas"
)

JANELA_MANUTENCAO_DIAS = 30
grupo_psd = tabela_mestra.groupby("id_psd", sort=False)

colunas_manutencao_30d = {
    "qtd_corretivas_emergenciais": "qtd_corretivas_emergenciais_30d",
    "qtd_corretivas_programadas": "qtd_corretivas_programadas_30d",
    "qtd_preventivas_nao_canceladas": "qtd_preventivas_nao_canceladas_30d",
}

for coluna_origem, coluna_nova in colunas_manutencao_30d.items():
    tabela_mestra[coluna_nova] = grupo_psd[coluna_origem].transform(
        lambda serie: serie.rolling(
            JANELA_MANUTENCAO_DIAS, min_periods=JANELA_MANUTENCAO_DIAS
        ).sum()
    )

colunas_historico_manutencao = [
    "dias_desde_ultima_corretiva",
    "dias_desde_ultima_preventiva",
    *colunas_manutencao_30d.values(),
]

auditoria_historico_manutencao = pd.DataFrame({
    "feature": colunas_historico_manutencao,
    "valores_nulos": [tabela_mestra[coluna].isna().sum() for coluna in colunas_historico_manutencao],
})

display(auditoria_historico_manutencao)

,feature,valores_nulos
0,dias_desde_ultima_corretiva,122
1,dias_desde_ultima_preventiva,1038
2,qtd_corretivas_emergenciais_30d,638
3,qtd_corretivas_programadas_30d,638
4,qtd_preventivas_nao_canceladas_30d,638


In [12]:
TARGET = "target_falha_7d"

features_categoricas = [
    "id_psd",
    "estacao_padrao",
    "plataforma_padrao",
    "dia_semana",
]

features_numericas = [
    "domingo_flag",
    "qtd_anomalias_criticas_30s",
    "qtd_criticas_pico_manha",
    "qtd_criticas_pico_tarde",
    "duracao_media_alarmes_seg",
    "duracao_maxima_alarme_seg",
    "qtd_portas_com_alarme",
    "qtd_anomalias_criticas_30s_7d",
    "qtd_criticas_pico_7d",
    "dias_com_anomalia_critica_7d",
    "duracao_maxima_alarme_7d",
    "proporcao_criticas_pico_7d",
    "dias_desde_ultima_corretiva",
    "dias_desde_ultima_preventiva",
    "qtd_corretivas_emergenciais_30d",
    "qtd_corretivas_programadas_30d",
    "qtd_preventivas_nao_canceladas_30d",
]

features_modelo = features_categoricas + features_numericas
colunas_controle = ["data_referencia", "qtd_falhas_proximos_7d", TARGET]
colunas_excluidas = sorted(set(tabela_mestra.columns) - set(features_modelo) - set(colunas_controle))

assert "qtd_falhas_confirmadas" not in features_modelo
assert "qtd_falhas_proximos_7d" not in features_modelo
assert TARGET not in features_modelo
assert "qtd_alarmes_total" not in features_modelo

resumo_features = pd.DataFrame({
    "grupo": ["categóricas", "numéricas", "total", "excluídas"],
    "quantidade": [
        len(features_categoricas),
        len(features_numericas),
        len(features_modelo),
        len(colunas_excluidas),
    ],
})

display(resumo_features)
print("Features do modelo:", features_modelo)
print("Colunas excluídas:", colunas_excluidas)

,grupo,quantidade
0,categóricas,4
1,numéricas,17
2,total,21
3,excluídas,18


Features do modelo: ['id_psd', 'estacao_padrao', 'plataforma_padrao', 'dia_semana', 'domingo_flag', 'qtd_anomalias_criticas_30s', 'qtd_criticas_pico_manha', 'qtd_criticas_pico_tarde', 'duracao_media_alarmes_seg', 'duracao_maxima_alarme_seg', 'qtd_portas_com_alarme', 'qtd_anomalias_criticas_30s_7d', 'qtd_criticas_pico_7d', 'dias_com_anomalia_critica_7d', 'duracao_maxima_alarme_7d', 'proporcao_criticas_pico_7d', 'dias_desde_ultima_corretiva', 'dias_desde_ultima_preventiva', 'qtd_corretivas_emergenciais_30d', 'qtd_corretivas_programadas_30d', 'qtd_preventivas_nao_canceladas_30d']
Colunas excluídas: ['dia_com_anomalia_critica', 'duracao_maxima_corretiva_horas', 'duracao_maxima_preventiva_horas', 'duracao_maxima_programada_horas', 'duracao_total_alarmes_seg', 'duracao_total_corretivas_horas', 'duracao_total_preventivas_horas', 'duracao_total_programadas_horas', 'qtd_alarmes_total', 'qtd_corretivas_emergenciais', 'qtd_corretivas_programadas', 'qtd_criticas_pico_dia', 'qtd_falhas_confirmadas'

In [13]:
colunas_historico_minimo = [
    *colunas_h1_h5,
    *colunas_manutencao_30d.values(),
]

mascara_alvo_disponivel = tabela_mestra[TARGET].notna()
mascara_historico_completo = tabela_mestra[colunas_historico_minimo].notna().all(axis=1)
mascara_base_modelagem = mascara_alvo_disponivel & mascara_historico_completo

base_modelagem = tabela_mestra.loc[
    mascara_base_modelagem,
    ["data_referencia", *features_modelo, TARGET],
].copy()

for coluna in features_categoricas:
    base_modelagem[coluna] = base_modelagem[coluna].astype(str)

base_modelagem[TARGET] = base_modelagem[TARGET].astype("int8")
base_modelagem = base_modelagem.sort_values(["data_referencia", "id_psd"]).reset_index(drop=True)

resumo_base_modelagem = pd.DataFrame({
    "situacao": [
        "linhas originais",
        "removidas sem alvo futuro completo",
        "removidas sem 30 dias de histórico",
        "linhas finais",
        "alvos positivos",
        "alvos negativos",
    ],
    "linhas": [
        len(tabela_mestra),
        int((~mascara_alvo_disponivel).sum()),
        int((mascara_alvo_disponivel & ~mascara_historico_completo).sum()),
        len(base_modelagem),
        int(base_modelagem[TARGET].sum()),
        int((base_modelagem[TARGET] == 0).sum()),
    ],
})

display(resumo_base_modelagem)
print(f"Período final: {base_modelagem['data_referencia'].min().date()} até {base_modelagem['data_referencia'].max().date()}")
print(f"Nulos mantidos nas features numéricas: {base_modelagem[features_numericas].isna().sum().sum():,}")

,situacao,linhas
0,linhas originais,24772
1,removidas sem alvo futuro completo,154
2,removidas sem 30 dias de histórico,638
3,linhas finais,23980
4,alvos positivos,11398
5,alvos negativos,12582


Período final: 2023-03-28 até 2026-03-21
Nulos mantidos nas features numéricas: 456


In [14]:
distribuicao_mensal_alvo = (
    base_modelagem
    .groupby(pd.Grouper(key="data_referencia", freq="MS"))
    .agg(
        linhas=(TARGET, "size"),
        positivos=(TARGET, "sum"),
        sistemas_psd=("id_psd", "nunique"),
    )
    .reset_index()
)
distribuicao_mensal_alvo["taxa_positivos"] = (
    distribuicao_mensal_alvo["positivos"]
    / distribuicao_mensal_alvo["linhas"]
    * 100
).round(2)

display(distribuicao_mensal_alvo)

,data_referencia,linhas,positivos,sistemas_psd,taxa_positivos
0,2023-03-01,88,58,22,65.91
1,2023-04-01,660,405,22,61.36
2,2023-05-01,682,348,22,51.03
3,2023-06-01,660,263,22,39.85
4,2023-07-01,682,308,22,45.16
5,2023-08-01,682,403,22,59.09
6,2023-09-01,660,399,22,60.45
7,2023-10-01,682,447,22,65.54
8,2023-11-01,660,468,22,70.91
9,2023-12-01,682,490,22,71.85


In [16]:
FIM_TREINO = pd.Timestamp("2025-03-24")
INICIO_VALIDACAO = pd.Timestamp("2025-04-01")
FIM_VALIDACAO = pd.Timestamp("2025-09-23")
INICIO_TESTE = pd.Timestamp("2025-10-01")

base_modelagem["conjunto"] = pd.Series(pd.NA, index=base_modelagem.index, dtype="string")
base_modelagem.loc[base_modelagem["data_referencia"] <= FIM_TREINO, "conjunto"] = "treino"
base_modelagem.loc[
    base_modelagem["data_referencia"].between(INICIO_VALIDACAO, FIM_VALIDACAO),
    "conjunto",
] = "validacao"
base_modelagem.loc[base_modelagem["data_referencia"] >= INICIO_TESTE, "conjunto"] = "teste"

base_treino = base_modelagem.loc[base_modelagem["conjunto"] == "treino"].copy()
base_validacao = base_modelagem.loc[base_modelagem["conjunto"] == "validacao"].copy()
base_teste = base_modelagem.loc[base_modelagem["conjunto"] == "teste"].copy()
base_intervalo_seguranca = base_modelagem.loc[base_modelagem["conjunto"].isna()].copy()

assert base_treino["data_referencia"].max() + pd.Timedelta(days=7) < base_validacao["data_referencia"].min()
assert base_validacao["data_referencia"].max() + pd.Timedelta(days=7) < base_teste["data_referencia"].min()

auditoria_divisao = (
    base_modelagem.dropna(subset=["conjunto"])
    .groupby("conjunto", as_index=False)
    .agg(
        data_inicial=("data_referencia", "min"),
        data_final=("data_referencia", "max"),
        linhas=(TARGET, "size"),
        positivos=(TARGET, "sum"),
        sistemas_psd=("id_psd", "nunique"),
    )
)
auditoria_divisao["taxa_positivos"] = (
    auditoria_divisao["positivos"] / auditoria_divisao["linhas"] * 100
).round(2)

print(f"Linhas removidas nos intervalos de segurança: {len(base_intervalo_seguranca):,}")
display(auditoria_divisao)

Linhas removidas nos intervalos de segurança: 308


,conjunto,data_inicial,data_final,linhas,positivos,sistemas_psd,taxa_positivos
0,teste,2025-10-01,2026-03-21,3784,1957,22,51.72
1,treino,2023-03-28,2025-03-24,16016,7264,22,45.35
2,validacao,2025-04-01,2025-09-23,3872,2019,22,52.14
